In [1]:
from can import Message

In [2]:
test = Message(data=[1, 2, 3, 4, 5])

In [3]:
test.data

bytearray(b'\x01\x02\x03\x04\x05')

In [4]:
test.dlc

5

In [5]:
print (test)

Timestamp:        0.000000    ID: 00000000    X Rx                DL:  5    01 02 03 04 05


In [17]:
import can

def send_pcan_fd_message():
    try:
        bus = can.Bus(
        interface='pcan',
        channel='PCAN_USBBUS1',
        fd=True,
        f_clock_mhz=30,
        nom_brp=12,
        nom_tseg1=3,
        nom_tseg2=1,
        nom_sjw=1,
        data_brp=3,
        data_tseg1=3,
        data_tseg2=1,
        data_sjw=1
)
        
        print("PCAN device connected successfully.")
    except can.CanError as e:
        print(f"Failed to connect to PCAN device: {e}")
        return

    # Create and send the message as usual...
    msg = can.Message(
        arbitration_id=0x123,
        data=[0x11, 0x22, 0x33, 0x44, 0x55, 0x66, 0x77, 0x88],
        is_extended_id=False,
        is_fd=True,
        bitrate_switch=True
    )
    bus.send(msg)
    bus.shutdown()

if __name__ == "__main__":
    send_pcan_fd_message()

PCAN device connected successfully.


In [20]:
"""
CAN FD Firmware Sender — Jupyter Notebook Version
===================================================
Copy this entire file into a single notebook cell, then
use the "USAGE" cell at the bottom to send firmware.
"""

import can
import struct
import time
import zlib
from pathlib import Path


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: Configuration — Edit these to match your setup
# ═══════════════════════════════════════════════════════════════════════════════

PCAN_CHANNEL    = 'PCAN_USBBUS1'
CMD_CAN_ID      = 0x03          # Command frames  (M-Board → S-Board)
DATA_CAN_ID     = 0x04          # Pure data frames (64 bytes, 512-bit aligned)
RESP_CAN_ID     = 0x05          # Responses        (S-Board → M-Board)
SBOARD_ID       = 0x01          # S-Board target ID

# PCAN FD bit timing (30 MHz clock, 500kbps nominal / 2Mbps data)
PCAN_FD_PARAMS = dict(
    f_clock_mhz = 30,
    nom_brp     = 12,
    nom_tseg1   = 3,
    nom_tseg2   = 1,
    nom_sjw     = 1,
    data_brp    = 3,
    data_tseg1  = 3,
    data_tseg2  = 1,
    data_sjw    = 1,
)

# Protocol
CMD_FW_TRANSFER_START   = 0x30
CMD_FW_IMAGE_HEADER     = 0x31
CMD_FW_IMAGE_COMPLETE   = 0x33
CMD_FW_TRANSFER_DONE    = 0x34
CMD_FW_START_BU_UPDATE  = 0x35
CMD_FW_START_SELF_UPD   = 0x36
CMD_FW_STATUS_REQUEST   = 0x37

CMD_FW_ACK              = 0x25
CMD_FW_NAK              = 0x26
CMD_FW_VERIFY_PASS      = 0x27
CMD_FW_VERIFY_FAIL      = 0x28

IMAGE_TYPE_BU           = 0x0000
IMAGE_TYPE_SBOARD       = 0x0001

BURST_SIZE              = 16
DATA_FRAME_SIZE         = 64
ACK_TIMEOUT             = 0.5
MAX_RETRIES             = 3
ERASE_TIMEOUT           = 5.0
VERIFY_TIMEOUT          = 5.0


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: Core functions — Run this cell once
# ═══════════════════════════════════════════════════════════════════════════════

def crc32_firmware(data: bytes) -> int:
    return zlib.crc32(data) & 0xFFFFFFFF


def pad_to_512bit(data: bytes) -> bytes:
    remainder = len(data) % DATA_FRAME_SIZE
    if remainder != 0:
        data += b'\xFF' * (DATA_FRAME_SIZE - remainder)
    return data


def build_cmd_frame(cmd, target_id, seq=0, payload=b''):
    header = struct.pack('<BBH', cmd, target_id, seq)
    frame_data = header + payload[:60].ljust(60, b'\x00')
    return can.Message(
        arbitration_id=CMD_CAN_ID, data=frame_data,
        is_extended_id=False, is_fd=True, bitrate_switch=True,
    )


def build_data_frame(data_64):
    assert len(data_64) == 64
    return can.Message(
        arbitration_id=DATA_CAN_ID, data=data_64,
        is_extended_id=False, is_fd=True, bitrate_switch=True,
    )


def parse_response(msg):
    if msg is None or len(msg.data) < 4:
        return None
    return {
        'cmd':     msg.data[0],
        'src_id':  msg.data[1],
        'seq':     struct.unpack_from('<H', msg.data, 2)[0],
        'payload': bytes(msg.data[4:]),
    }


class FirmwareSender:

    def __init__(self, channel=PCAN_CHANNEL):
        print(f"[INIT] Connecting to {channel}...")
        self.bus = can.Bus(
            interface='pcan', channel=channel, fd=True, **PCAN_FD_PARAMS,
        )
        print(f"[INIT] Connected.")

    def close(self):
        self.bus.shutdown()
        print("[INIT] Disconnected.")

    def send(self, msg):
        self.bus.send(msg)

    def wait_for_response(self, expected_cmd=None, timeout=ACK_TIMEOUT):
        deadline = time.monotonic() + timeout
        while time.monotonic() < deadline:
            remaining = deadline - time.monotonic()
            msg = self.bus.recv(timeout=max(remaining, 0.001))
            if msg is None or msg.arbitration_id != RESP_CAN_ID:
                continue
            resp = parse_response(msg)
            if resp is None:
                continue
            if expected_cmd is not None and resp['cmd'] != expected_cmd:
                continue
            return resp
        return None

    # ── Protocol ──────────────────────────────────────────────────────────

    def send_transfer_start(self):
        print("\n[START] Sending TRANSFER_START (S-Board will erase staging banks)...")
        self.send(build_cmd_frame(CMD_FW_TRANSFER_START, SBOARD_ID))
        resp = self.wait_for_response(expected_cmd=CMD_FW_ACK, timeout=ERASE_TIMEOUT)
        if resp:
            print("  ✓ S-Board ready")
            return True
        print(f"  ✗ No ACK (timeout {ERASE_TIMEOUT}s)")
        return False

    def send_image_header(self, image_type, image_data, version=1):
        padded = pad_to_512bit(image_data)
        image_crc = crc32_firmware(image_data)
        total_frames = len(padded) // DATA_FRAME_SIZE
        dest_bank = 0x0A0000 if image_type == IMAGE_TYPE_BU else 0x0C0000
        name = "BU" if image_type == IMAGE_TYPE_BU else "S-Board"

        print(f"\n[HEADER] {name} firmware:")
        print(f"  Size:   {len(image_data)} → {len(padded)} bytes (padded)")
        print(f"  Frames: {total_frames}  |  CRC: 0x{image_crc:08X}  |  Ver: {version}")

        payload = struct.pack('<HHIIIIII',
            0x4601, image_type, len(padded), image_crc,
            version, dest_bank, 0x080000, total_frames,
        )
        self.send(build_cmd_frame(CMD_FW_IMAGE_HEADER, SBOARD_ID, seq=0, payload=payload))
        resp = self.wait_for_response(expected_cmd=CMD_FW_ACK, timeout=ACK_TIMEOUT)
        if resp:
            print("  ✓ Header ACK'd")
            return True
        print("  ✗ No ACK")
        return False

    def send_image_data(self, image_data):
        padded = pad_to_512bit(image_data)
        total_frames = len(padded) // DATA_FRAME_SIZE
        total_bursts = (total_frames + BURST_SIZE - 1) // BURST_SIZE

        print(f"\n[DATA] {len(padded)} bytes → {total_frames} frames → {total_bursts} bursts")

        frame_idx = 0
        while frame_idx < total_frames:
            burst_start = frame_idx
            frames_in_burst = min(BURST_SIZE, total_frames - frame_idx)

            for i in range(frames_in_burst):
                offset = frame_idx * DATA_FRAME_SIZE
                self.send(build_data_frame(padded[offset : offset + DATA_FRAME_SIZE]))
                frame_idx += 1

            last_idx = frame_idx - 1
            ack_ok = False

            for retry in range(MAX_RETRIES):
                resp = self.wait_for_response(expected_cmd=CMD_FW_ACK, timeout=ACK_TIMEOUT)
                if resp and resp['seq'] == last_idx:
                    ack_ok = True
                    break
                else:
                    print(f"  ⟳ Retry burst {burst_start//BURST_SIZE+1} ({retry+1}/{MAX_RETRIES})")
                    for i in range(frames_in_burst):
                        offset = (burst_start + i) * DATA_FRAME_SIZE
                        self.send(build_data_frame(padded[offset : offset + DATA_FRAME_SIZE]))

            if not ack_ok:
                print(f"\n  ✗ Burst failed after {MAX_RETRIES} retries")
                return False

            burst_num = burst_start // BURST_SIZE + 1
            pct = frame_idx * 100 // total_frames
            print(f"  Burst {burst_num}/{total_bursts} — {pct}%", end='\r')

        print(f"\n  ✓ All {total_frames} frames sent")
        return True

    def send_image_complete(self):
        print(f"\n[VERIFY] Requesting CRC verification...")
        self.send(build_cmd_frame(CMD_FW_IMAGE_COMPLETE, SBOARD_ID))
        resp = self.wait_for_response(expected_cmd=None, timeout=VERIFY_TIMEOUT)
        if resp and resp['cmd'] == CMD_FW_VERIFY_PASS:
            print("  ✓ CRC PASSED")
            return True
        print("  ✗ CRC FAILED or no response")
        return False

    def send_transfer_done(self):
        print(f"\n[DONE] Finalizing...")
        self.send(build_cmd_frame(CMD_FW_TRANSFER_DONE, SBOARD_ID))
        resp = self.wait_for_response(expected_cmd=CMD_FW_ACK, timeout=ACK_TIMEOUT)
        if resp:
            print("  ✓ Transfer complete")
            return True
        print("  ✗ No ACK")
        return False

    def send_one_image(self, image_type, image_data, version=1):
        if not self.send_image_header(image_type, image_data, version):
            return False
        if not self.send_image_data(image_data):
            return False
        if not self.send_image_complete():
            return False
        return True

    def trigger_bu_update(self):
        print(f"\n[TRIGGER] START_BU_UPDATE...")
        self.send(build_cmd_frame(CMD_FW_START_BU_UPDATE, SBOARD_ID))
        resp = self.wait_for_response(expected_cmd=CMD_FW_ACK, timeout=ACK_TIMEOUT)
        print("  ✓ Started" if resp else "  ✗ No ACK")
        return resp is not None

    def trigger_self_update(self):
        print(f"\n[TRIGGER] START_SELF_UPDATE...")
        self.send(build_cmd_frame(CMD_FW_START_SELF_UPD, SBOARD_ID))
        resp = self.wait_for_response(expected_cmd=CMD_FW_ACK, timeout=ACK_TIMEOUT)
        print("  ✓ Started" if resp else "  ✗ No ACK")
        return resp is not None


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: USAGE — Edit paths and run
# ═══════════════════════════════════════════════════════════════════════════════

# ── Step 1: Set your firmware paths ──────────────────────────────────────────
BU_BIN_PATH  = r"C:\Users\parthasarathy.m\bu_firmware.bin"        # ← EDIT THIS
SB_BIN_PATH = r"C:\Users\parthasarathy.m\sboard_firmware.bin"   # ← Uncomment if sending S-Board image too

# ── Step 2: Load binary ─────────────────────────────────────────────────────
bu_data = Path(BU_BIN_PATH).read_bytes()
print(f"Loaded BU: {len(bu_data)} bytes ({len(bu_data)/1024:.1f} KB)")

# sb_data = Path(SB_BIN_PATH).read_bytes()          # ← Uncomment if needed
# print(f"Loaded SB: {len(sb_data)} bytes ({len(sb_data)/1024:.1f} KB)")

# ── Step 3: Connect and send ─────────────────────────────────────────────────
sender = FirmwareSender()

try:
    sender.send_transfer_start()
    sender.send_one_image(IMAGE_TYPE_BU, bu_data, version=1)
    # sender.send_one_image(IMAGE_TYPE_SBOARD, sb_data, version=1)  # ← Uncomment if needed
    sender.send_transfer_done()

    # Optional: trigger the update
    # sender.trigger_bu_update()
    # sender.trigger_self_update()
finally:
    sender.close()

Loaded BU: 83768 bytes (81.8 KB)
[INIT] Connecting to PCAN_USBBUS1...
[INIT] Connected.

[START] Sending TRANSFER_START (S-Board will erase staging banks)...
  ✗ No ACK (timeout 5.0s)

[HEADER] BU firmware:
  Size:   83768 → 83776 bytes (padded)
  Frames: 1309  |  CRC: 0xE7A8F0A6  |  Ver: 1
  ✗ No ACK

[DONE] Finalizing...
  ✗ No ACK
[INIT] Disconnected.


**Without Acknowledgement**

In [25]:
"""
CAN FD Firmware Sender — Fire & Forget (No ACK)
=================================================
Streams a raw .bin file as 64-byte CAN FD data frames.
Each frame = 512 bits = one flash write on the receiver.

For Jupyter notebook — just edit the path and run.
"""

import can
import struct
import time
import zlib
from pathlib import Path


# ─── Configuration ────────────────────────────────────────────────────────────

PCAN_CHANNEL    = 'PCAN_USBBUS1'
DATA_CAN_ID     = 0x04          # Pure 64-byte data frames
DATA_FRAME_SIZE = 64            # 512 bits = 1 flash block

# Delay between frames (seconds) — tune based on S-Board flash write speed
# 0     = full speed (may overflow RX FIFO)
# 0.001 = 1ms per frame (~64 KB/s)
# 0.005 = 5ms per frame (~12.8 KB/s, safe for flash writes)
INTER_FRAME_DELAY = 0.001

# PCAN FD bit timing (30 MHz clock, 500kbps nom / 2Mbps data)
PCAN_FD_PARAMS = dict(
    f_clock_mhz = 30,
    nom_brp     = 12,
    nom_tseg1   = 3,
    nom_tseg2   = 1,
    nom_sjw     = 1,
    data_brp    = 3,
    data_tseg1  = 3,
    data_tseg2  = 1,
    data_sjw    = 1,
)


# ─── Functions ────────────────────────────────────────────────────────────────

def pad_to_512bit(data: bytes) -> bytes:
    """Pad to multiple of 64 bytes with 0xFF (erased flash state)."""
    remainder = len(data) % DATA_FRAME_SIZE
    if remainder != 0:
        data += b'\xFF' * (DATA_FRAME_SIZE - remainder)
    return data


def send_bin(bin_path: str, channel=PCAN_CHANNEL, delay=INTER_FRAME_DELAY):
    """Load a .bin file and stream it as 64-byte CAN FD frames."""

    # Load
    raw = Path(bin_path).read_bytes()
    padded = pad_to_512bit(raw)
    total_frames = len(padded) // DATA_FRAME_SIZE
    crc = zlib.crc32(raw) & 0xFFFFFFFF

    print(f"File:    {Path(bin_path).name}")
    print(f"Size:    {len(raw)} bytes ({len(raw)/1024:.1f} KB)")
    print(f"Padded:  {len(padded)} bytes → {total_frames} frames")
    print(f"CRC32:   0x{crc:08X}")
    print(f"Delay:   {delay*1000:.1f} ms/frame")
    print()

    # Connect
    bus = can.Bus(interface='pcan', channel=channel, fd=True, **PCAN_FD_PARAMS)
    print(f"Connected to {channel}")

    # Send
    t_start = time.monotonic()

    try:
        for i in range(total_frames):
            offset = i * DATA_FRAME_SIZE
            chunk = padded[offset : offset + DATA_FRAME_SIZE]

            msg = can.Message(
                arbitration_id=DATA_CAN_ID,
                data=chunk,
                is_extended_id=False,
                is_fd=True,
                bitrate_switch=True,
            )
            bus.send(msg)

            if delay > 0:
                time.sleep(delay)

            # Progress every 50 frames
            if (i + 1) % 50 == 0 or i == total_frames - 1:
                pct = (i + 1) * 100 // total_frames
                kb = (i + 1) * DATA_FRAME_SIZE / 1024
                print(f"  Frame {i+1}/{total_frames} — {kb:.1f} KB — {pct}%", end='\r')

        elapsed = time.monotonic() - t_start
        throughput = len(padded) / elapsed / 1024

        print(f"\n\nDone! {total_frames} frames in {elapsed:.2f}s ({throughput:.1f} KB/s)")
        print(f"CRC32: 0x{crc:08X}  ← verify this on S-Board side")

    finally:
        bus.shutdown()
        print("Disconnected.")


# ─── USAGE — Edit path and run ────────────────────────────────────────────────

BIN_PATH = r"C:\Users\parthasarathy.m\sboard_firmware.bin"       # ← EDIT THIS

send_bin(BIN_PATH)

File:    sboard_firmware.bin
Size:    83768 bytes (81.8 KB)
Padded:  83776 bytes → 1309 frames
CRC32:   0xE7A8F0A6
Delay:   1.0 ms/frame

Connected to PCAN_USBBUS1
  Frame 1309/1309 — 81.8 KB — 100%

Done! 1309 frames in 2.70s (30.3 KB/s)
CRC32: 0xE7A8F0A6  ← verify this on S-Board side
Disconnected.
